## Notebook 2:<br> Extractor Implementation and Qualitative Debugging

    In this notebook, we implement an Intelligent Document Processing (IDP) pipeline that transforms unstructured phone transcriptions into structured JSON payloads.

### Notebook Workflow
1. **General Extractor Architecture:** Inspect the core components  of a extractor with `systemprompt` and a simplified `Pydantic schema`.

2. **Configurate the Baseline Extractor:** Use a high-capability LLM chatbot to build a  Pydantic schema and prompt based on the test orders.<br> Run the extractor. 

3. **Qualitative Debugging Loop:** Use the same chatbot to evaluate extractor results and as a thought partner to analyze failure cases, refine field definitions, and improve extraction coverage.<br>Run the improved extractor. Repeat until you are satisfied with the extractor. 

4. **Overfitting Check:** Validate the refined schema against an unseen evaluation dataset (Dataset 2).



In [ ]:
# Inspect available test order files
from pathlib import Path

folder = Path.cwd().parent / "data" / "test_orders"
for file in folder.iterdir():
    if file.is_file():
        print(file.name)

In [ ]:
# Load the sample orders as test_orders
from pizza_order_extractor.helper_functions import load_orders_as_list

test_order_file = "<your_test_order_file>.json" #use the one of your  sample order sets for the extractor configuration, and the other set later for validation

test_orders = load_orders_as_list(test_order_file)
for index, order in enumerate(test_orders, 1):
    print(f"{index}: {order}\n")

## 1. General Extractor Architecture

Our extraction chain consists of two primary components:
1. **`Pydantic Schema`:** Defines target fields, data types, and validation rules.


2. **`System Prompt`:** Instructs the model on mapping rules while isolating system instructions from user inputs.

We chain these components using **LangChain Expression Language (LCEL)**:
```python
extractor = prompt_template | structured_llm
```

Below is a minimal baseline extraction setup with a very simple schema that defines only a few scalar fields (pizza name, size, toppings)<br> and a generic field (special requests) for anything that doesn’t fit anywhere else. Run this extractor with your sample orders. 



In [ ]:
from typing import List, Optional, Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from pizza_order_extractor.helper_functions import (
    check_apikeys,
    load_orders_as_list,
    save_extraction_to_file
)

check_apikeys()


class PizzaItem(BaseModel):
    """Information about an ordered pizza."""
    pizza_name: Optional[str] = Field(
        default=None, description="The name of the pizza if provided."
    )
    pizza_size: Literal["small", "medium", "large"] = Field(
        ..., description="Size of the pizza."
    )
    toppings: list[str]  = Field(
        default_factory=list, 
        description="List of toppings."
    )
    special_instructions: list[str]  = Field(
        default_factory=list,
        description="Use ONLY for requests that cannot fit standard fields "
    )

# Use the name of this class to provide the schema to the model: schema = ExtractedPizzaOrder
class ExtractedPizzaOrder(BaseModel):
    """Container for extracted items."""
    order: List[PizzaItem]


system_prompt = """
You are an expert extraction algorithm.
Extract relevant information from the transcript.
"""


def run_extractor(schema, system_prompt, model_name, provider, file_name):
    model = init_chat_model(model_name, model_provider=provider, temperature=0.0)
    
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("user", "{order_text}"),
    ])
    
    structured_llm = model.with_structured_output(schema=schema, method="json_schema")
    extractor = prompt_template | structured_llm
    
    test_orders = load_orders_as_list(file_name)
    response_list = []
    
    for i, order in enumerate(test_orders, 1):
        response = extractor.invoke({"order_text": order})
        response_list.append(response)
        print(f"--- Order {i} ---\n{order}")
        print(f"--- Extracted Output {i} ---\n{response}\n")
        
    save_extraction_to_file(response_list, model_name)
    return response_list #returns a list of pydantic objects


baseline_results = run_extractor(
    schema=ExtractedPizzaOrder,
    system_prompt=system_prompt,
    model_name="<your_antropic/openai...model_name here>",
    provider="<paste_the_provider_name_here",
    file_name=test_order_file
)

**Observation:**

Click on the `link` provided to open and inspect the results.

Note that the “Special Requests” field contains a variety of categories (depending on your examples, you’ll find a mix of categories such as: “Warning: Seafood Allergy,” “half without onions,” “extra cheese on the other half,” “thin crust”).

If someone in the kitchen receives this information, they can likely use it to prepare the correct pizza. However, this extraction is not yet suitable for machine processing. We need to figure out how to structure the schema in greater detail so that the fields become more granular. If we were to structure the extractor manually, we would proceed as follows: identify topics that frequently appear in the “Special Requests” section and create a separate field for each piece of information.

However, we skip many of these steps: When you ask a chatbot for output that is as structured as possible, you usually receive a much more detailed schema right away.

The question for production use is: How detailed does the extraction need to be?

**For our tutorial, our goal is clear: to keep the “Special Requests” field as small as possible.**





## 2. Configurate the `Baseline Extractor`

### Task 1: Use a powerful web chatbot to configurate the Baseline Extractor
1. Copy the `10 sample orders`  into a proprietary web chatbot (e.g., Claude, ChatGPT, or Gemini).
2. Ask the chatbot to propose a modular `Pydantic schema` with nested structures.
3. Ask the chatbot to refine the `system prompt` to explicitly address non-linear speech corrections (e.g., *"no wait, scratch that"*).
4. Ask for explanation until you understand the solution proposed by the chatbot. 
4. Implement your updated schema and prompt below and run it. 


In [ ]:
#  paste your pydantic schema here





In [ ]:
# paste your system prompt here
system_promt = """
paste_your_systemprompt_here
"""

In [ ]:
# run the extractor, 

baseline_results = run_extractor(
    schema=paste_your_classname_here
    system_prompt=system_prompt,
    model_name="paste_your_modelname_here",
    provider="paste_the_modelprovider_here",
    file_name=test_order_file
)


## 3. Qualitative Debugging & Schema Refinement Loop

### Task 2: Refine Schema & System Prompt
1. Copy the sample orders and the extraction results into the chatbot conversation, ask the chatbot to evaluate the extraction,<br> discuss improvments, implement the improvment and run the extractor. 


2. Repite this until you dont get further improvent. 


In [ ]:
# paste your improved schema here


In [ ]:
# paste the improved system prompt here
system_prompt = """

"""

In [ ]:
# run the extractor, make sure that the schema name is updated. 

baseline_results = run_extractor(
    schema=paste_your_classname_here
    system_prompt=system_prompt,
    model_name="paste_your_modelname_here",
    provider="paste_the_modelprovider_here",
    file_name=test_order_file
)

## 4. Generalization Test & Overfitting Detection

When iterating on prompts against a small dataset ($N=10$), there is a risk of **overfitting**—tuning prompt rules so tightly to specific phrasing in Dataset 1 that accuracy drops on unseen orders.

### Task 3: Test Validation Dataset 2
1. Run your refined pipeline on `validation_orders` (generated in Notebook 1).
2. Copy the orders and results to the Chatbot and use the same prompt to generate a evaluacion. Compare error rates between Dataset 1 and Dataset 2. If the results are ok, proceed with point 3, else try to improve the extractor.
3. Once verified, copy your production schema to `schemas.py` and your prompt to `prompts.py` in src/pizza_order_extractor. We will use them in notebook 4 for benchmarking with diferent models.

In [ ]:
# Inspect available test order files
import os
from pathlib import Path

folder = Path.cwd().parent / "data" / "test_orders"
for file in folder.iterdir():
    if file.is_file():
        print(file.name)

In [ ]:
# Load the sample orders as test_orders
from pizza_order_extractor.helper_functions import load_orders_as_list

#test_order_file = "paste_your_second_dataset_here.json"
validation_order_file = "your_second_sample_order_set.json"
validation_orders = load_orders_as_list(validation_order_file)

for index, order in enumerate(validation_orders, 1):
    print(f"{index}: {order}\n")

In [ ]:
# run the extractor

baseline_results = run_extractor(
    schema=paste_your_classname_here
    system_prompt=system_prompt,
    model_name="paste_your_modelname_here",
    provider="paste_the_modelprovider_here",
    file_name=test_order_file
)

## Don´t forget: 
1. Open `src\pizza_order_extractor\schemas.py` and save your pydantic schema there.
2. Open `src\pizza_order_extractor\prompts.py` and saver your system prompt there.